## Final dataset preparation
- Now, ones we have done cleaning and feature engineering, let's perform some final tasks :
1. Remove outliers
2. Encode categorical variables (if any)
3. standarize / normalize the data
4. Remove unnecessary columns for feeding into models

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Important thorughout this project
%matplotlib inline


In [ ]:
df = pd.read_csv('../data/engineered/satellites_engineered.csv')

In [ ]:
df.head()

----
### 1. Removing outliers
##### Key question : Why even remove outliers if we are going to perform anomaly detection?


### 1️⃣ Garbage-in → Garbage-out

* ML models (Isolation Forest, KMeans, etc.) assume that most of your data is “normal” to learn patterns.
* If your dataset accidentally has **obvious errors** (e.g., height = 0 km, speed = 1e6 m/s, negative values), the model may learn wrong patterns or consider normal data as anomalous.

---

### 2️⃣ Helps define “normal”

* Anomaly detection models define anomalies relative to what is normal.
* If there are **obvious outliers**, your baseline “normal” distribution will be skewed, reducing detection accuracy for true anomalies like orbital maneuvers.

---

### 3️⃣ Prevents false positives

* Spotting obvious errors ensures you **don’t flag bad data as anomalies**.
* Real anomalies should reflect **interesting satellite behavior**, not just bad CSV entries or measurement glitches.

---

✅ **Summary:**

* Spotting anomalies early is about **data quality**, not about defeating your anomaly detection goal.
* Once the dataset is clean, your ML models can focus on **real, subtle anomalies** (like unusual delta in orbital height, speed, or maneuvers).




---

## ❗For prototyping purpose, we will not perform outlier removal step for now, later on we may perform that!

### 2. Encode categorical variables 

In [ ]:
df.info()

- We only have one categorical feature, that's going to be feed into the models and that is 'satellite type'.
- So we will use OneHotEncoder to encode the feature

In [ ]:
from sklearn.preprocessing import OneHotEncoder

In [ ]:
# Fit and transform
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_encoded = encoder.fit_transform(df[['SAT_TYPE']])

# Get actual category names
encoded_cols = encoder.get_feature_names_out(['SAT_TYPE'])

# Convert to DataFrame with proper column names
df_encoded = pd.concat([df.drop('SAT_TYPE', axis=1), pd.DataFrame(X_encoded, columns=encoded_cols)], axis=1)


In [ ]:
df_encoded.head()

In [ ]:
# We combined Main dataframe (excluding the SAT_TYPE) + (X encoded (values) + encoded_cols feature names) 
X_encoded

In [ ]:
encoded_cols

In [ ]:
df_encoded.info()

---
### 3. Remove irrelevent features

- We will now remove those features that will NOT be used in any of the ML models, that are :
1. OBJECT_NAME         
2. OBJECT_ID            
3. EPOCH  (only will be USEFUL in 'time series analysis' or Delta features like changes in speed at successive epochs), but not right now for protype
4. EPHEMERIS_TYPE   (all have just a single value : 11990)    
5. CLASSIFICATION_TYPE (all have just a single value : U)
6. NORAD_CAT_ID (unique identifier)
7. ELEMENT_SET_NO  (unique identifier)

In [ ]:
df_encoded = df_encoded.drop(columns=['OBJECT_NAME', 'OBJECT_ID', 'EPOCH', 'EPHEMERIS_TYPE', 'CLASSIFICATION_TYPE', 'NORAD_CAT_ID', 'ELEMENT_SET_NO'])

In [ ]:
df_encoded.info()

---
### 4. Feature Scaling 

- **Numeric Features:**  
  Features like `SEMI_MAJOR_AXIS`, `ORBIT_HEIGHT`, `ORBITAL_SPEED`, etc., have different units and ranges. Scaling them using **StandardScaler** standardizes the values (mean=0, std=1) so that all numeric features contribute equally to distance-based algorithms like **KMeans** and **Isolation Forest**.

- **Categorical Features (One-Hot Encoded):**  
  Features like `SAT_TYPE` are converted into binary columns (0/1). These **do not require scaling**, as their values are already normalized and scaling would distort the categorical meaning.

- **Key Idea:**  
  - Scale numeric continuous features.  
  - Keep one-hot categorical features as-is.  
  This ensures the model correctly interprets distances and patterns without bias from different units.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer # lets you apply different preprocessing to different columns

In [ ]:
scaler = StandardScaler()

encoded_cols = [col for col in df_encoded.columns if col.startswith('SAT_TYPE_')]
numeric_cols = df_encoded.drop(columns = encoded_cols).columns

ct = ColumnTransformer([
    ('scaler', StandardScaler(), numeric_cols), # scale numeric feature
    ('pass', 'passthrough', encoded_cols) # keep encoded columns as is
])

df_scaled = ct.fit_transform(df_encoded)

In [ ]:
df_scaled

In [ ]:
# convert to dataframe                        # numeric_col is a series, so convert to a list 
df_scaled = pd.DataFrame(df_scaled, columns = numeric_cols.tolist() + encoded_cols)

In [ ]:
df_scaled.head()

- Save as model ready dataset

In [ ]:
df_scaled.to_csv('../data/model_ready/sattelites_final.csv', index=False)

#### Now we are ready for Machine Learning models!